In [ ]:
from functions.utilities import preprocess
import pandas as pd
import plotly.express as px

In [ ]:
filename='processed_data\itigi_comments_2024.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))
id_to_author = litigi.set_index('id')['author'].to_dict()
litigi['parent_author'] = litigi['parent_id'].str.split('_').str[1].map(id_to_author)
litigi['num_words'] = litigi['body_preprocessed'].str.split().apply(len)
litigi.head()

In [ ]:
from sentence_transformers import SentenceTransformer

# Sample data
texts = litigi['body_preprocessed'].tolist()


model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

text_embeddings = model.encode(texts,show_progress_bar=True)


In [ ]:
import numpy as np

all_embeddings = np.vstack([text_embeddings])
labels = litigi["author"].tolist()


In [ ]:
from torch.utils.tensorboard import SummaryWriter
import torch
import os

log_dir = "./runs/embedding_projector"
os.makedirs(log_dir, exist_ok=True)

writer = SummaryWriter(log_dir)

# Write embeddings
writer.add_embedding(
    torch.tensor(all_embeddings),
    metadata=labels,
    tag="sentence_embeddings"
)

writer.close()
